# CAISO LMP vs Actual Generation Analysis

This notebook analyzes the relationship between Locational Marginal Prices (LMP) and actual energy generation in the CAISO grid. The core hypothesis being explored is the **merit-order effect**: as renewable generation increases (near-zero marginal cost), it displaces higher-cost thermal units, pushing system LMP lower.

**Data sources:**
- `1_day_LMP_DAM.csv` — Day-Ahead Market LMP by node (2025-12-10)
- `1_month_CAISO_load.csv` — Actual system load (December 2025)
- `1_month_Renewables_Act.csv` — Actual renewable generation by hub and type (December 2025)

**Key analyses:**
1. LMP time series and decomposition (MCE / MCC / MCL / MGHG)
2. Trading hub price comparison (NP15 / SP15 / ZP26)
3. LMP vs. total system load
4. LMP vs. renewable generation (merit-order effect)
5. Renewable penetration and price suppression

## 1. Setup

In [ ]:
%mamba install pandas

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from datetime import datetime

## 2. Load Data

In [ ]:
# LMP data — Day-Ahead Market, 2025-12-10
df_lmp = pd.read_csv("1_day_LMP_DAM.csv")

# Actual system load — December 2025
df_actual = pd.read_csv("1_month_CAISO_load.csv")

# Actual renewable generation — December 2025
df_renewable = pd.read_csv("1_month_Renewables_Act.csv")

print(f"LMP rows:       {len(df_lmp):,}")
print(f"Actual rows:    {len(df_actual):,}")
print(f"Renewable rows: {len(df_renewable):,}")

## 3. Parsing & Quality Checks

In [ ]:
def parse_dates(df):
    """Parse CAISO GMT interval timestamps and set as index."""
    df['start_time'] = pd.to_datetime(df['INTERVALSTARTTIME_GMT'], utc=True, errors='coerce')
    df['end_time']   = pd.to_datetime(df['INTERVALENDTIME_GMT'],   utc=True, errors='coerce')
    df = df.set_index('start_time').sort_index()
    print(f"  Missing indexes: {df.index.isna().sum()}")
    df['hour']       = df.index.hour
    df['day']        = df.index.day
    df['weekday']    = df.index.weekday
    df['day_of_week']= df.index.strftime('%a')
    return df


def time_series_checks(df, label):
    """Basic consistency checks for a time-indexed DataFrame."""
    print(f"--- {label} ---")
    print(f"  Inferred freq:       {pd.infer_freq(df.index)}")
    print(f"  Duplicate timestamps:{df.index.duplicated().sum()}")
    nan_runs = df['MW'].isna().astype(int).groupby(df['MW'].notna().cumsum()).sum()
    nan_runs = nan_runs[nan_runs > 0]
    print(f"  NaN runs: {nan_runs.to_dict() if len(nan_runs) else 'none'}\n")

In [ ]:
print("Parsing LMP...")
df_lmp = parse_dates(df_lmp)

print("Parsing actual load...")
df_actual = parse_dates(df_actual)

print("Parsing renewables...")
df_renewable = parse_dates(df_renewable)

In [ ]:
time_series_checks(df_lmp,      "LMP")
time_series_checks(df_actual,   "Actual Load")
time_series_checks(df_renewable,"Renewables")

## 4. LMP Analysis

### 4a. LMP structure

Each row in the LMP dataset represents one node × one interval × one price component.

| Component | Meaning |
| --- | --- |
| **LMP** | Total Locational Marginal Price |
| **MCE** | Marginal Cost of Energy (system lambda) |
| **MCC** | Marginal Cost of Congestion |
| **MCL** | Marginal Cost of Losses |
| **MGHG** | Marginal GHG compliance cost |

In [ ]:
print("LMP types:", df_lmp['LMP_TYPE'].unique())
print("\nSample nodes:")
print(df_lmp['NODE'].unique()[:10])

### 4b. Pivot to wide format and isolate trading hubs

CAISO trading hubs aggregate nodal prices for three geographic zones:
- **NP15** — Northern California (PG&E)
- **SP15** — Southern California (SCE / SDG&E)
- **ZP26** — Central Valley (PG&E south)

In [ ]:
# Pivot so each LMP_TYPE becomes a column, keeping node granularity
df_wide = df_lmp.pivot_table(
    index=['start_time', 'NODE'],
    columns='LMP_TYPE',
    values='MW',
    aggfunc='first'
).reset_index()

# Verify decomposition: LMP ≈ MCE + MCC + MCL + MGHG
df_wide['LMP_recalc'] = df_wide[['MCE', 'MCC', 'MCL', 'MGHG']].fillna(0).sum(axis=1)
df_wide['decomp_error'] = (df_wide['LMP_recalc'] - df_wide['LMP']).abs()
print(f"Max decomposition error: {df_wide['decomp_error'].max():.6f} $/MWh")
print(df_wide.head(3))

In [ ]:
# Identify hub nodes (NP15, SP15, ZP26) — CAISO hub node names contain the hub acronym
HUB_KEYWORDS = {'NP15': 'NP15', 'SP15': 'SP15', 'ZP26': 'ZP26'}

def assign_hub(node_str):
    for hub, kw in HUB_KEYWORDS.items():
        if kw in str(node_str):
            return hub
    return None

df_wide['hub'] = df_wide['NODE'].apply(assign_hub)
df_hubs = df_wide[df_wide['hub'].notna()].copy()

print("Hub nodes found:")
print(df_hubs.groupby('hub')['NODE'].unique())

In [ ]:
# Average across nodes within each hub and interval
df_hub_ts = (
    df_hubs
    .groupby(['start_time', 'hub'])[['LMP', 'MCE', 'MCC', 'MCL', 'MGHG']]
    .mean()
    .reset_index()
    .set_index('start_time')
)

print(df_hub_ts.head(6))
print(f"\nTime range: {df_hub_ts.index.min()} → {df_hub_ts.index.max()}")

### 4c. Hub LMP time series

In [ ]:
fig, ax = plt.subplots(figsize=(13, 4))

for hub, grp in df_hub_ts.groupby('hub'):
    ax.plot(grp.index, grp['LMP'], label=hub, linewidth=1.5)

ax.set_title('Day-Ahead LMP by Trading Hub — 2025-12-10 (UTC)')
ax.set_ylabel('LMP ($/MWh)')
ax.xaxis.set_major_formatter(mdates.DateFormatter('%H:%M'))
ax.grid(True, axis='y', alpha=0.4)
ax.legend(title='Hub')
fig.autofmt_xdate()
plt.tight_layout()
plt.show()

### 4d. LMP component decomposition (system-average)

In [ ]:
# Use NP15 as a representative hub; fall back to all hubs if NP15 is unavailable
available_hubs = df_hub_ts['hub'].unique()
ref_hub = 'NP15' if 'NP15' in available_hubs else available_hubs[0]
df_ref = df_hub_ts[df_hub_ts['hub'] == ref_hub].copy()

components = ['MCE', 'MCC', 'MCL', 'MGHG']
colors = ['#2196F3', '#FF9800', '#4CAF50', '#9C27B0']

fig, ax = plt.subplots(figsize=(13, 4))

ax.stackplot(
    df_ref.index,
    [df_ref[c].fillna(0) for c in components],
    labels=components,
    colors=colors,
    alpha=0.75
)
ax.plot(df_ref.index, df_ref['LMP'], color='black', linewidth=1.2, linestyle='--', label='Total LMP')

ax.set_title(f'LMP Decomposition — {ref_hub} Hub (2025-12-10, UTC)')
ax.set_ylabel('$/MWh')
ax.xaxis.set_major_formatter(mdates.DateFormatter('%H:%M'))
ax.grid(True, axis='y', alpha=0.4)
ax.legend(loc='upper left')
fig.autofmt_xdate()
plt.tight_layout()
plt.show()

## 5. Prepare Actual Load & Renewable Generation

The LMP data covers 2025-12-10 only, so we slice the monthly load and renewable data to match that day for the joint analysis.

In [ ]:
# --- Actual CAISO-wide load (hourly) ---
df_caiso_load = df_actual[df_actual['TAC_AREA_NAME'] == 'CA ISO-TAC'].copy()
load_hourly = (
    df_caiso_load
    .pivot_table(index=df_caiso_load.index, values='MW', aggfunc='sum')
    .rename(columns={'MW': 'load_MW'})
    .round(2)
)

# Slice to LMP day
load_day = load_hourly.loc['2025-12-10'].copy()

print(f"Load shape (full month): {load_hourly.shape}")
print(f"Load shape (LMP day):    {load_day.shape}")
print(load_day.head())

In [ ]:
# --- Renewable generation (5-min or 15-min, resample to hourly) ---
renew_hourly = (
    df_renewable
    .resample('1h')['MW']
    .sum()
    .rename('renew_MW')
    .round(2)
)

renew_day = renew_hourly.loc['2025-12-10'].copy()

# Renewable by type for the LMP day
renew_by_type_day = (
    df_renewable.loc['2025-12-10']
    .groupby(['RENEWABLE_TYPE'])['MW']
    .resample('1h')
    .sum()
    .unstack(level=0)
    .fillna(0)
)

print(f"Renewable types: {df_renewable['RENEWABLE_TYPE'].unique()}")
print(renew_day.head())

In [ ]:
# Build system-average LMP (mean across all three hubs, hourly)
lmp_system = (
    df_hub_ts
    .groupby(level='start_time')[['LMP', 'MCE', 'MCC', 'MCL', 'MGHG']]
    .mean()
    .round(4)
)

# Resample to hourly (DAM is already hourly, but in case of 5-min intervals)
lmp_hourly = lmp_system.resample('1h').mean().round(4)

print(f"Hourly LMP range: {lmp_hourly.index.min()} → {lmp_hourly.index.max()}")
print(lmp_hourly.head())

## 6. Merge LMP with Generation Data

In [ ]:
df_merged = (
    lmp_hourly
    .join(load_day, how='inner')
    .join(renew_day, how='left')
)

# Renewable penetration as % of total load
df_merged['renew_pct'] = (df_merged['renew_MW'] / df_merged['load_MW'] * 100).round(2)

# Net load = total load minus renewables
df_merged['net_load_MW'] = (df_merged['load_MW'] - df_merged['renew_MW']).round(2)

print(f"Merged rows: {len(df_merged)}")
print(df_merged)

## 7. LMP vs Generation — Visualizations

### 7a. Dual-axis: LMP and System Load over the day

In [ ]:
fig, ax1 = plt.subplots(figsize=(13, 5))
ax2 = ax1.twinx()

ax1.plot(df_merged.index, df_merged['LMP'], color='#E53935', linewidth=2, label='Avg Hub LMP')
ax2.fill_between(df_merged.index, df_merged['load_MW'], alpha=0.25, color='#1E88E5', label='System Load')
ax2.plot(df_merged.index, df_merged['load_MW'], color='#1E88E5', linewidth=1.2, linestyle='--')

ax1.set_ylabel('LMP ($/MWh)', color='#E53935')
ax2.set_ylabel('Load (MW)', color='#1E88E5')
ax1.set_title('CAISO LMP vs System Load — 2025-12-10 (UTC)')
ax1.tick_params(axis='y', labelcolor='#E53935')
ax2.tick_params(axis='y', labelcolor='#1E88E5')

ax1.xaxis.set_major_formatter(mdates.DateFormatter('%H:%M'))
ax1.grid(True, axis='y', alpha=0.3)

lines1, labels1 = ax1.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax1.legend(lines1 + lines2, labels1 + labels2, loc='upper left')

fig.autofmt_xdate()
plt.tight_layout()
plt.show()

### 7b. Dual-axis: LMP and Renewable Generation over the day

In [ ]:
fig, ax1 = plt.subplots(figsize=(13, 5))
ax2 = ax1.twinx()

ax1.plot(df_merged.index, df_merged['LMP'], color='#E53935', linewidth=2, label='Avg Hub LMP')
ax2.fill_between(df_merged.index, df_merged['renew_MW'], alpha=0.25, color='#43A047', label='Renewables')
ax2.plot(df_merged.index, df_merged['renew_MW'], color='#43A047', linewidth=1.2, linestyle='--')

ax1.set_ylabel('LMP ($/MWh)', color='#E53935')
ax2.set_ylabel('Renewable Generation (MW)', color='#43A047')
ax1.set_title('CAISO LMP vs Renewable Generation — 2025-12-10 (UTC)\n(Merit-order effect: renewables↑ → LMP↓)')
ax1.tick_params(axis='y', labelcolor='#E53935')
ax2.tick_params(axis='y', labelcolor='#43A047')

ax1.xaxis.set_major_formatter(mdates.DateFormatter('%H:%M'))
ax1.grid(True, axis='y', alpha=0.3)

lines1, labels1 = ax1.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax1.legend(lines1 + lines2, labels1 + labels2, loc='upper left')

fig.autofmt_xdate()
plt.tight_layout()
plt.show()

### 7c. Scatter: LMP vs Renewable Penetration (%)

In [ ]:
fig, ax = plt.subplots(figsize=(7, 5))

sc = ax.scatter(
    df_merged['renew_pct'],
    df_merged['LMP'],
    c=df_merged.index.hour,
    cmap='plasma',
    s=80,
    edgecolors='white',
    linewidths=0.5,
    zorder=3
)

# Linear trend line
valid = df_merged[['renew_pct', 'LMP']].dropna()
if len(valid) >= 2:
    z = np.polyfit(valid['renew_pct'], valid['LMP'], 1)
    p = np.poly1d(z)
    xs = np.linspace(valid['renew_pct'].min(), valid['renew_pct'].max(), 100)
    ax.plot(xs, p(xs), 'k--', linewidth=1.2, label=f'Trend: slope={z[0]:.2f} $/MWh per %')
    ax.legend()

plt.colorbar(sc, ax=ax, label='Hour of day (UTC)')
ax.set_xlabel('Renewable Penetration (% of load)')
ax.set_ylabel('Avg Hub LMP ($/MWh)')
ax.set_title('LMP vs Renewable Penetration — 2025-12-10')
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

### 7d. Scatter: LMP vs Net Load (Load − Renewables)

In [ ]:
fig, ax = plt.subplots(figsize=(7, 5))

sc = ax.scatter(
    df_merged['net_load_MW'],
    df_merged['LMP'],
    c=df_merged.index.hour,
    cmap='viridis',
    s=80,
    edgecolors='white',
    linewidths=0.5,
    zorder=3
)

# Annotate each point with its UTC hour
for _, row in df_merged.iterrows():
    if pd.notna(row['net_load_MW']) and pd.notna(row['LMP']):
        ax.annotate(
            str(_.hour),
            (row['net_load_MW'], row['LMP']),
            textcoords='offset points', xytext=(4, 3),
            fontsize=7, color='gray'
        )

# Trend line
valid = df_merged[['net_load_MW', 'LMP']].dropna()
if len(valid) >= 2:
    z = np.polyfit(valid['net_load_MW'], valid['LMP'], 1)
    p = np.poly1d(z)
    xs = np.linspace(valid['net_load_MW'].min(), valid['net_load_MW'].max(), 100)
    ax.plot(xs, p(xs), 'r--', linewidth=1.2, label=f'Trend: slope={z[0]:.4f} $/MWh per MW')
    ax.legend()

plt.colorbar(sc, ax=ax, label='Hour of day (UTC)')
ax.set_xlabel('Net Load (MW)  [Load − Renewables]')
ax.set_ylabel('Avg Hub LMP ($/MWh)')
ax.set_title('LMP vs Net Load — 2025-12-10\n(Higher net load → more thermal dispatch → higher prices)')
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

### 7e. Stacked generation vs LMP — overview panel

In [ ]:
fig, (ax_top, ax_bot) = plt.subplots(2, 1, figsize=(13, 8), sharex=True)

# ---- Top panel: LMP by hub ----
hub_colors = {'NP15': '#1565C0', 'SP15': '#E53935', 'ZP26': '#558B2F'}
for hub, grp in df_hub_ts.groupby('hub'):
    grp_hourly = grp['LMP'].resample('1h').mean()
    ax_top.plot(grp_hourly.index, grp_hourly,
                label=hub, color=hub_colors.get(hub), linewidth=1.8)

ax_top.set_ylabel('LMP ($/MWh)')
ax_top.set_title('CAISO Trading Hub LMP vs Generation — 2025-12-10 (UTC)')
ax_top.legend(title='Hub')
ax_top.grid(True, axis='y', alpha=0.3)

# ---- Bottom panel: stacked renewables + net load line ----
if not renew_by_type_day.empty:
    renew_types = renew_by_type_day.columns.tolist()
    ax_bot.stackplot(
        renew_by_type_day.index,
        [renew_by_type_day[t] for t in renew_types],
        labels=renew_types,
        alpha=0.7
    )

ax_bot.plot(load_day.index, load_day['load_MW'],
            color='black', linewidth=1.5, linestyle='--', label='Total Load')
ax_bot.set_ylabel('MW')
ax_bot.set_xlabel('Hour (UTC)')
ax_bot.legend(title='Generation Type', loc='upper left', fontsize=8)
ax_bot.grid(True, axis='y', alpha=0.3)
ax_bot.xaxis.set_major_formatter(mdates.DateFormatter('%H:%M'))

fig.autofmt_xdate()
plt.tight_layout()
plt.show()

## 8. Summary Statistics

In [ ]:
summary = df_merged[['LMP', 'MCE', 'MCC', 'MCL', 'load_MW', 'renew_MW', 'renew_pct', 'net_load_MW']].describe().round(2)
print(summary)

In [ ]:
# Correlation matrix
corr_cols = ['LMP', 'load_MW', 'renew_MW', 'renew_pct', 'net_load_MW']
corr = df_merged[corr_cols].corr().round(3)
print("Pearson correlation matrix:")
print(corr)

In [ ]:
# Peak / off-peak LMP split (on-peak = hours 7–22 local; UTC−8 in December → 15–06 UTC)
# Using a simple heuristic: on-peak if CAISO load > median load
median_load = df_merged['load_MW'].median()
df_merged['peak_flag'] = df_merged['load_MW'] > median_load

peak_lmp    = df_merged.loc[df_merged['peak_flag'],  'LMP'].mean()
offpeak_lmp = df_merged.loc[~df_merged['peak_flag'], 'LMP'].mean()

peak_renew    = df_merged.loc[df_merged['peak_flag'],  'renew_pct'].mean()
offpeak_renew = df_merged.loc[~df_merged['peak_flag'], 'renew_pct'].mean()

print(f"On-peak  avg LMP: {peak_lmp:.2f} $/MWh  |  avg renewable %: {peak_renew:.1f}%")
print(f"Off-peak avg LMP: {offpeak_lmp:.2f} $/MWh  |  avg renewable %: {offpeak_renew:.1f}%")
print(f"\nLMP premium (on vs off-peak): {peak_lmp - offpeak_lmp:+.2f} $/MWh")